# Ultralytics YOLOv8 Training Notebook

## This Section is for Split Data

In [2]:
import os
import shutil
import numpy as np
from collections import Counter
from skmultilearn.model_selection import iterative_train_test_split

# === CONFIGURATION ===
dataset_path = "../data/Afif"           # main folder containing 'images', 'labels', and 'data.yaml'
output_path = "../data/dataset_split"   # output folder for the split dataset
train_ratio = 0.85                      # proportion of training set

images_path = os.path.join(dataset_path, "images")
labels_path = os.path.join(dataset_path, "labels")
yaml_src = os.path.join(dataset_path, "data.yaml")
yaml_dst = os.path.join(output_path, "data.yaml")

# Get all image files
image_files = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
image_files.sort()

# Read labels and build multi-label matrix
all_classes = set()
labels_matrix = []

for img_file in image_files:
    label_file = os.path.splitext(img_file)[0] + ".txt"
    label_path = os.path.join(labels_path, label_file)
    classes_for_img = set()

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) > 0:
                    classes_for_img.add(int(parts[0]))

    all_classes.update(classes_for_img)
    labels_matrix.append(classes_for_img)

# Map class IDs to matrix indices
class_list = sorted(list(all_classes))
class_to_idx = {cls: idx for idx, cls in enumerate(class_list)}

# Convert to binary matrix
Y = np.zeros((len(image_files), len(class_list)), dtype=int)
for i, classes_for_img in enumerate(labels_matrix):
    for cls in classes_for_img:
        Y[i, class_to_idx[cls]] = 1

# Iterative stratification split
X = np.array(image_files).reshape(-1, 1)
X_train, y_train, X_val, y_val = iterative_train_test_split(X, Y, test_size=1-train_ratio)

train_files = [x[0] for x in X_train]
val_files = [x[0] for x in X_val]

def copy_files(file_list, split_type):
    """Copy images and labels in YOLO folder structure."""
    img_out_dir = os.path.join(output_path, "images", split_type)
    lbl_out_dir = os.path.join(output_path, "labels", split_type)
    os.makedirs(img_out_dir, exist_ok=True)
    os.makedirs(lbl_out_dir, exist_ok=True)

    for img_file in file_list:
        shutil.copy2(os.path.join(images_path, img_file), os.path.join(img_out_dir, img_file))
        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_src = os.path.join(labels_path, label_file)
        if os.path.exists(label_src):
            shutil.copy2(label_src, os.path.join(lbl_out_dir, label_file))

def count_class_distribution(file_list):
    """Count bounding boxes per class."""
    counter = Counter()
    for img_file in file_list:
        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_path = os.path.join(labels_path, label_file)
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) > 0:
                        counter[int(parts[0])] += 1
    return counter

# Copy datasets in YOLO format
copy_files(train_files, "train")
copy_files(val_files, "val")

# Copy data.yaml if exists
if os.path.exists(yaml_src):
    os.makedirs(output_path, exist_ok=True)
    shutil.copy2(yaml_src, yaml_dst)
    print(f"📄 data.yaml copied to {yaml_dst}")
else:
    print("⚠️ No data.yaml found in dataset_path.")

# Count distributions
train_dist = count_class_distribution(train_files)
val_dist = count_class_distribution(val_files)

print(f"✅ Dataset split into YOLO format using Iterative Stratification.")
print(f"Training set: {len(train_files)} images, Validation set: {len(val_files)} images.\n")

print("📊 Class distribution (bounding boxes per class):")
for cls in class_list:
    print(f"Class {cls}: Train={train_dist.get(cls, 0)}, Val={val_dist.get(cls, 0)}")

📄 data.yaml copied to ../data/dataset_split/data.yaml
✅ Dataset split into YOLO format using Iterative Stratification.
Training set: 456 images, Validation set: 81 images.

📊 Class distribution (bounding boxes per class):
Class 0: Train=135, Val=34
Class 1: Train=303, Val=58
Class 2: Train=103, Val=17
Class 3: Train=162, Val=29
Class 4: Train=112, Val=10
Class 5: Train=106, Val=23
Class 6: Train=130, Val=22
Class 7: Train=193, Val=27
Class 8: Train=138, Val=43
Class 9: Train=98, Val=11
Class 10: Train=154, Val=26
Class 11: Train=156, Val=20
Class 12: Train=93, Val=16
Class 13: Train=116, Val=26
Class 14: Train=99, Val=21
Class 15: Train=425, Val=123
Class 16: Train=64, Val=9


## This Section is for Color Transfer

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # go one level up

from utils import normalizer, reset, validation

In [ ]:
reset.delete_all_jpg_files("/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/BulkNormalizedAnnotatedData")

In [ ]:
rfc: str = "data/IMG00425.JPG"
mtd: normalizer.TransferMethod="mean_std"

In [ ]:
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images","data/BulkNormalizedAnnotatedData/images",transfer_method=mtd)
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images/train","data/BulkNormalizedAnnotatedData/images/train",transfer_method=mtd)
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images/val","data/BulkNormalizedAnnotatedData/images/val",transfer_method=mtd)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # go one level up

from utils import normalizer, reset, validation

In [ ]:
reset.delete_all_jpg_files("/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/BulkNormalizedAnnotatedData")

In [ ]:
rfc: str = "data/IMG00425.JPG"
mtd: normalizer.TransferMethod="mean_std"

In [ ]:
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images","data/BulkNormalizedAnnotatedData/images",transfer_method=mtd)
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images/train","data/BulkNormalizedAnnotatedData/images/train",transfer_method=mtd)
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images/val","data/BulkNormalizedAnnotatedData/images/val",transfer_method=mtd)

## This Section is for Training Yolo

In [1]:
from ultralytics import YOLO
import optuna
import time
import os

/Users/ahmadfariz/Projects/pythonAI/MARROWS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### insert data in the data section below

In [2]:
from ultralytics.data.utils import check_det_dataset


dataset= "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/data.yaml"
check_det_dataset(dataset)

{'names': {0: 'Poli Eritroblast',
  1: 'Orto Eritroblast',
  2: 'Baso Eritroblast',
  3: 'Myeloblast',
  4: 'Progranulocyte',
  5: 'Myelocyte',
  6: 'Metamyelocyte',
  7: 'Band',
  8: 'Segment',
  9: 'Basofil',
  10: 'Limfoblast',
  11: 'Limfosit',
  12: 'MKs',
  13: 'Plasma cell',
  14: 'Monosit',
  15: 'Monoblast',
  16: 'Dummy'},
 'nc': 17,
 'path': PosixPath('../data/Layer2Combine'),
 'train': '/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train',
 'val': '/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/val',
 'yaml_file': '/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/data.yaml',
 'channels': 3}

#### Check data integrity section below

In [3]:
from PIL import Image
import os

def check_jpeg_corruption(folder):
    index = 0
    for root, _, files in os.walk(folder):
        for fname in files:
            if fname.lower().endswith(".jpg") or fname.lower().endswith(".jpeg"):
                path = os.path.join(root, fname)
                print(f"Checking: {path}")
                try:
                    with Image.open(path) as img:
                        # img.verify()  # Check integrity
                        img.load()
                        print(f"Valid: {path}")
                        index += 1
                except Exception as e:
                    print(f"Corrupted: {path} – {e}")
    print(f"Total valid images: {index}")

check_jpeg_corruption("/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train")
check_jpeg_corruption("/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/val")

Checking: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train/IMG03200.JPG
Valid: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train/IMG03200.JPG
Checking: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train/IMG06422.JPG
Valid: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train/IMG06422.JPG
Checking: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train/IMG06344.JPG
Valid: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train/IMG06344.JPG
Checking: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train/IMG06436.JPG
Valid: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train/IMG06436.JPG
Checking: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train/IMG06393.JPG
Valid: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Layer2Combine/images/train/IMG06393.JPG
Checking: /Users/ahmadfar

#### ♻️ (Optional): Re-save all images to make sure no Corrupt File (May Cause Instability in data probably i am not sure either) DO NOT DO THIS MORE THAN ONCE

In [6]:
from PIL import Image
import os
def resave_all_images(folder):
    for root, _, files in os.walk(folder):
        for fname in files:
            if fname.lower().endswith(('.jpg', '.jpeg')):
                path = os.path.join(root, fname)
                try:
                    img = Image.open(path)
                    img.load()
                    img.convert("RGB").save(path, "JPEG", quality=95)
                    print(f"Resaved: {path}")
                except Exception as e:
                    print(f"Skip: {path} — {e}")


# resave_all_images("/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/train")
resave_all_images("/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val")

Resaved: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val/IMG06185.JPG
Resaved: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val/IMG06026.JPG
Resaved: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val/IMG06027.JPG
Resaved: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val/IMG06033.JPG
Resaved: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val/IMG03177.JPG
Resaved: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val/IMG06392.JPG
Resaved: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val/IMG03203.JPG
Resaved: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val/IMG06435.JPG
Resaved: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val/IMG06145.JPG
Resaved: /Users/ahmadfariz/Projects/pythonAI/MARROWS/data/dataset_split/images/val/IMG06186.JPG
Resaved: /Users/ahmadfariz/Projects/pyth

In [ ]:
import yaml
import optuna
from ultralytics import YOLO
import torch
import shutil
import os
import json

def train_model(hyp, trial_num, use_default=False):
    trial_name = f"trial_{trial_num}"
    project_dir = f"runs/train/{trial_name}"

    # Hapus direktori sebelumnya jika ada
    if os.path.exists(project_dir):
        shutil.rmtree(project_dir)

    hyp_path = None
    if not use_default:
        hyp_path = f"{trial_name}_hyp.yaml"
        with open(hyp_path, 'w') as f:
            yaml.dump(hyp, f)

    # Gunakan device Apple MPS jika tersedia
    # device = "mps" if torch.backends.mps.is_available() else "cpu"
    device = "cpu"

    # Load model
    model = YOLO("yolo11n-seg.pt")
    # # Jika menggunakan model kustom, pastikan file YAML ada
    # model = YOLO("../mods/yolov11_cbam.yaml")

    # Train
    results = model.train(
        data=dataset,
        epochs=100,
        imgsz=640,
        batch=8,
        name=trial_name,
        cfg=hyp_path if hyp_path else None,
        patience=15,  # Early stopping
        device=device,
    )

    # Simpan semua hasil
    try:
        result_dir = os.path.join("runs/train", trial_name)

        # Salin best.pt ke lokasi terpisah jika ingin
        best_weight = os.path.join(result_dir, "weights", "best.pt")
        if os.path.exists(best_weight):
            shutil.copy(best_weight, f"{trial_name}_best.pt")

        # Simpan metrics.json sebagai dict
        metrics_json = os.path.join(result_dir, "metrics.json")
        if os.path.exists(metrics_json):
            with open(metrics_json, 'r') as f:
                metrics_dict = json.load(f)
        else:
            metrics_dict = {}

        # Simpan confusion matrix
        cm_file = os.path.join(result_dir, "confusion_matrix.png")
        if os.path.exists(cm_file):
            shutil.copy(cm_file, f"{trial_name}_confusion_matrix.png")

        # Simpan CSV results
        csv_file = os.path.join(result_dir, "results.csv")
        if os.path.exists(csv_file):
            shutil.copy(csv_file, f"{trial_name}_results.csv")

        # Return mAP@50 jika tersedia
        return metrics_dict.get("metrics/mAP50(B)", 0.0)

    except Exception as e:
        print(f"⚠️ Error saving results for trial {trial_num}: {e}")
        return 0.0


def objective(trial):
    if trial.number == 0:
        print("🚀 Running baseline trial with default YOLOv11 hyperparameters...")
        return train_model(None, trial.number, use_default=True)

    hyp = {
        "lr0": trial.suggest_float('lr0', 1e-5, 1e-1, log=True),
        "momentum": trial.suggest_float('momentum', 0.80, 0.99),
        "weight_decay": trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
        "box": trial.suggest_float('box', 0.02, 0.4),
        "cls": trial.suggest_float('cls', 0.2, 1.0),
        "hsv_h": trial.suggest_float('hsv_h', 0.0, 0.1),
        "hsv_s": trial.suggest_float('hsv_s', 0.0, 0.7),
        "hsv_v": trial.suggest_float('hsv_v', 0.0, 0.4),
    }

    try:
        return train_model(hyp, trial.number)
    except Exception as e:
        print(f"⚠️ Trial {trial.number} failed: {e}")
        return 0.0

In [5]:
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Using device: {device}")

Using device: mps


In [ ]:
storage = optuna.storages.RDBStorage("sqlite:///yolov8_tuning.db")

study = optuna.create_study(
    direction="maximize",
    study_name="YOLOv8_Tuning",
    storage=storage,
    load_if_exists=True
)

# Kamu bisa menjalankan ulang cell ini berkali-kali dan proses akan dilanjutkan
study.optimize(objective, n_trials=15)

NameError: name 'optuna' is not defined

In [ ]:
best_trial = study.best_trial
best_trial_number = best_trial.number
best_model_path = f"runs/detect/trial_{best_trial.number}/weights/best.pt"

In [ ]:
best_model = YOLO(best_model_path)
val_results = best_model.val(data=dataset, conf=0.25)

In [ ]:
print(f"Best trial number: {best_trial_number}")
print(f"Best model path: {best_model_path}")

## This Section is for Loop A Model for Validations

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # go one level up

from utils import normalizer, reset, validation 
from ultralytics import YOLO

In [ ]:
#replace with best(detect) or segment
selection = "segment"

In [ ]:
model = YOLO(f"runs/detect/afif_{selection}/weights/best.pt")

### Loop A Model for Multiple Data Variant Prediction

In [ ]:
subset_list = ["Sparse", "Normal", "Clumpped", "CombineTest"]
# subset_list = ["CombineTest"]

for subset in subset_list:
    results = model.predict(
        source=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/{subset}/images/val",
        save=True,
        save_txt=True,
        save_conf=True,
        project=f"runs/detect/afif_{selection}",
        name=f"predict{subset}",
        exist_ok=True
    )

### Loop A Model for Multiple Validatio Data Variant Annotation vs Prediciton Count

In [ ]:

subset_list = ["Sparse", "Normal", "Clumpped","CombineTest"]

for subset in subset_list:
    validation.compare_annotation_vs_prediction(
        gt_label_folder=f"runs/detect/afif_{selection}/predict{subset}/actual_labels",  # Ground truth
        pred_label_folder=f"runs/detect/afif_{selection}/predict{subset}/labels",  # prediction
        output_img_path=f"{selection}_loss_ratio_plot_{subset}.png"
    )

### Loop A Model for Multiple Validation Data Variant

In [ ]:
# from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

model_path = "best.pt"
subset_paths = {
    "Sparse": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Sparse",
    "Normal": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Normal",
    "Clumpped": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Clumpped",
    "CombineTest": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/CombineTest"
}
yaml_paths = {k: os.path.join(v, "data.yaml") for k, v in subset_paths.items()}

summary_data = []
roc_data = []

for subset_name in subset_paths.keys():
    print(f"🔍 Evaluating: {subset_name}")
    
    # Run model validation
    results = model.val(data=yaml_paths[subset_name],save_json=True , split=f"val", save=False)

   

### Loop A Model for Multiple Data Variant ROC Analysis

In [ ]:
# subset_list = ["Sparse", "Normal", "Clumpped"]
# subset_list = ["CombineTest"]
subset_list = ["Sparse", "Normal", "Clumpped", "CombineTest"]

for subset in subset_list:
    print(f"\n🚀 Processing subset: {subset}")
    validation.run_roc_analysis(
        yaml_path=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/{subset}/data.yaml",
        gt_folder=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/{subset}/labels/val",
        pred_folder=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/notebooks/runs/detect/afif_{selection}/predict{subset}/labels",
        subset_name=subset,
        selection=selection,
    )